# 🛠️ AI IT Helpdesk Agent
**Capabilities:** Agent + RAG + Tools + SQL
**LLM:** Ollama (running locally inside this Colab notebook)

Diagnoses common IT issues using a knowledge base (RAG), reasons about the fix using a locally-hosted Ollama LLM (Agent), and creates support tickets in a SQL (SQLite) database when it can't resolve an issue (Tools).

Run every cell below **in order, top to bottom**. If Colab disconnects (new day / long gap), just run all cells again from the top — Ollama and the model will need to be reinstalled/re-pulled each fresh session.

In [1]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 11 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (651 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


## Step 1: Install Ollama

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Step 2: Start the Ollama Server (runs in background)

In [3]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(6)
print("Ollama server started.")

Ollama server started.


## Step 3: Pull a Lightweight Model

`llama3.2:1b` is used here as a lightweight, resource-efficient model — ideal for local deployment where fast inference and low memory footprint matter (e.g., real-time helpdesk responses without GPU dependency). The same pipeline can be scaled to larger models such as `llama3.2` or `phi3` for higher-accuracy responses in production environments with more compute available.

In [4]:
!ollama pull llama3.2:1b

## Step 4: Test Ollama Connection

In [5]:
import requests

OLLAMA_MODEL = "llama3.2:1b"

response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": OLLAMA_MODEL, "prompt": "Hello, are you working?", "stream": False}
)

print(response.json()["response"])

Hello. This is an artificial intelligence model, and I'm available to help you 24/7. I don't have a traditional work schedule, but I'm here and ready to assist you with any questions or tasks you have, whether it's during business hours or outside of them. How can I help you today?


## Step 5: Build Knowledge Base (RAG data source)

In [6]:
import pandas as pd

data = {
    "issue": [
        "Cannot reset password",
        "VPN not connecting",
        "Printer offline / not printing",
        "Laptop running very slow",
        "Software installation failed",
        "Email not syncing / not receiving mail",
        "WiFi keeps disconnecting",
        "Blue screen error / system crash",
        "Cannot access shared drive",
        "Zoom/Teams audio or video not working",
        "Forgot laptop login password",
        "System asking for software license activation",
        "Outlook not opening",
        "USB device not detected",
        "Screen flickering / display issue"
    ],
    "category": [
        "Account", "Network", "Hardware", "Performance", "Software",
        "Email", "Network", "System", "Access", "Software",
        "Account", "Software", "Software", "Hardware", "Hardware"
    ],
    "solution_steps": [
        "Go to company portal > Forgot Password > verify OTP sent to registered email/mobile > set new password.",
        "Check internet connection > restart VPN client > verify correct VPN server address > reinstall VPN client if issue persists.",
        "Check printer power and cable > restart printer > remove and re-add printer in Settings > update printer driver.",
        "Close unused background apps > run disk cleanup > check startup programs > restart laptop > scan for malware.",
        "Run installer as Administrator > check disk space > disable antivirus temporarily > redownload installer if corrupted.",
        "Check internet connection > verify correct email settings (IMAP/POP) > remove and re-add account > clear cache.",
        "Restart router > forget and reconnect WiFi network > update WiFi driver > move closer to router.",
        "Note error code shown > restart in Safe Mode > update drivers > run Windows Memory Diagnostic > contact IT if repeats.",
        "Check network connection > verify correct permissions with admin > reconnect mapped drive > restart file explorer.",
        "Check camera/mic permissions in app settings > update app > restart app > check device audio/video settings.",
        "Use 'Forgot Password' option on login screen > verify identity via registered email > reset via IT admin if locked out.",
        "Go to Settings > Activation > enter valid license key > ensure internet connection is active > contact IT if key invalid.",
        "Restart Outlook in Safe Mode > repair Office installation > check PST file size > create new Outlook profile.",
        "Try different USB port > update USB drivers > restart laptop > test device on another system.",
        "Update graphics driver > check display cable connection > adjust refresh rate in display settings > test with external monitor."
    ]
}

df = pd.DataFrame(data)
df.to_csv("it_issues.csv", index=False)
print(df.head())
print(f"\nTotal issues in knowledge base: {len(df)}")

                            issue     category  \
0           Cannot reset password      Account   
1              VPN not connecting      Network   
2  Printer offline / not printing     Hardware   
3        Laptop running very slow  Performance   
4    Software installation failed     Software   

                                      solution_steps  
0  Go to company portal > Forgot Password > verif...  
1  Check internet connection > restart VPN client...  
2  Check printer power and cable > restart printe...  
3  Close unused background apps > run disk cleanu...  
4  Run installer as Administrator > check disk sp...  

Total issues in knowledge base: 15


## Step 6: RAG Retrieval Logic (TF-IDF similarity search)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

kb = pd.read_csv("it_issues.csv")

vectorizer = TfidfVectorizer()
issue_vectors = vectorizer.fit_transform(kb["issue"])

def retrieve_relevant_issue(user_query, top_k=1):
    query_vector = vectorizer.transform([user_query])
    similarities = cosine_similarity(query_vector, issue_vectors).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    idx = top_indices[0]
    return {
        "issue": kb.iloc[idx]["issue"],
        "category": kb.iloc[idx]["category"],
        "solution": kb.iloc[idx]["solution_steps"],
        "confidence": round(similarities[idx], 2)
    }

# quick test
print(retrieve_relevant_issue("my wifi keeps dropping"))

{'issue': 'WiFi keeps disconnecting', 'category': 'Network', 'solution': 'Restart router > forget and reconnect WiFi network > update WiFi driver > move closer to router.', 'confidence': np.float64(0.82)}


## Step 7: Agent Reasoning (Ollama LLM + RAG combined)

In [8]:
def call_ollama(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
    )
    return response.json()["response"]

def ai_helpdesk_agent(user_query):
    retrieved = retrieve_relevant_issue(user_query)

    if retrieved["confidence"] < 0.3:
        return {"response": None, "resolved": False, "retrieved": retrieved}

    prompt = f"""You are a professional IT helpdesk agent.
A user reported this issue: \"{user_query}\"

Relevant knowledge base info:
Issue: {retrieved['issue']}
Category: {retrieved['category']}
Solution steps: {retrieved['solution']}

Respond in a friendly, professional tone. Explain the issue diagnosis briefly, then give the solution steps clearly as a numbered list."""

    answer = call_ollama(prompt)

    return {
        "response": answer,
        "resolved": True,
        "retrieved": retrieved
    }

# quick test
result = ai_helpdesk_agent("my wifi keeps dropping")
print(result["response"])

I'd be happy to help you troubleshoot the issue with your WiFi connectivity. Based on the information you provided, it appears that your WiFi is experiencing frequent disconnections. I'll take a moment to summarize the issue diagnosis and then provide a clear and concise solution step-by-step to help you resolve the issue.

**Issue Diagnosis:**

Your WiFi is experiencing frequent disconnections, which may be due to a variety of factors such as:

* Poor internet signal strength or interference
* Physical obstructions or distance from the router
* Device issues or compatibility problems
* Outdated WiFi drivers or firmware

**Solution Steps:**

Here's a step-by-step guide to help you resolve the issue:

1. **Restart the router:** This is the most straightforward solution. Simply unplug the power cord, wait for 30 seconds, and plug it back in to restart the router. This will reset the router to its default settings and may resolve any temporary issues.

2. **Forget and reconnect the WiFi n

## Step 8: Tool Calling (SQLite Ticket Database + Escalation + System Status)
Uses a **SQL database (SQLite)** to store tickets — covers the SQL requirement your college asked for.

In [9]:
import sqlite3
from datetime import datetime

DB_FILE = "tickets.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS tickets (
            ticket_id TEXT PRIMARY KEY,
            timestamp TEXT,
            issue TEXT,
            status TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

def create_ticket(issue_description):
    ticket_id = f"TCK-{int(datetime.now().timestamp())}"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO tickets (ticket_id, timestamp, issue, status) VALUES (?, ?, ?, ?)",
        (ticket_id, timestamp, issue_description, "Escalated to IT Team")
    )
    conn.commit()
    conn.close()
    return ticket_id

def get_all_tickets():
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query("SELECT * FROM tickets", conn)
    conn.close()
    return df

def check_system_status():
    return "All systems operational"

def run_helpdesk_agent(user_query):
    print(f"User: {user_query}\n")
    result = ai_helpdesk_agent(user_query)

    if result["resolved"]:
        print("Agent Response:\n")
        print(result["response"])
        print(f"\nMatch confidence: {result['retrieved']['confidence']}")
    else:
        ticket_id = create_ticket(user_query)
        print("Agent: I couldn't find a confident match in our knowledge base for this issue.")
        print(f"I've escalated this to our IT team. Your ticket ID is: {ticket_id}")

    print(f"\nSystem Status: {check_system_status()}")

# test known issue
run_helpdesk_agent("printer is not printing anything")
print("\n" + "="*50 + "\n")
# test unknown issue -> should escalate
run_helpdesk_agent("my laptop keyboard keys are sticky and some letters don't type")
print("\n" + "="*50 + "\n")
print("All tickets in database:")
print(get_all_tickets())

User: printer is not printing anything

Agent Response:

Hello, I'd be happy to help you troubleshoot the issue with your printer.

Based on the information provided by the user, it appears that the printer is offline or not printing. To diagnose the issue, I'll need to ask a few more questions to narrow down the possible causes. Can you please provide me with the following information:

1. What type of printer do you have (e.g., HP, Canon, Epson)?
2. Have you recently updated your printer drivers or firmware?
3. Have you checked the printer power and cable to ensure they are properly connected?
4. Have you tried restarting the printer?

Once I have this information, I'll be happy to provide you with a step-by-step solution to resolve the issue.

In the meantime, here are the solution steps to follow:

1. Check the printer power: Ensure the printer is turned on and the power cord is securely connected to both the printer and the wall outlet.
2. Check the printer cable: Verify that the 

## Step 9: Create the Streamlit App File (Frontend + Backend, using Ollama + SQL)

In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sqlite3
from datetime import datetime

OLLAMA_MODEL = "llama3.2:1b"

kb = pd.read_csv("it_issues.csv")
vectorizer = TfidfVectorizer()
issue_vectors = vectorizer.fit_transform(kb["issue"])

DB_FILE = "tickets.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS tickets (
            ticket_id TEXT PRIMARY KEY,
            timestamp TEXT,
            issue TEXT,
            status TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

def call_ollama(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
    )
    return response.json()["response"]

def retrieve_relevant_issue(user_query, top_k=1):
    query_vector = vectorizer.transform([user_query])
    similarities = cosine_similarity(query_vector, issue_vectors).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    idx = top_indices[0]
    return {
        "issue": kb.iloc[idx]["issue"],
        "category": kb.iloc[idx]["category"],
        "solution": kb.iloc[idx]["solution_steps"],
        "confidence": round(similarities[idx], 2)
    }

def create_ticket(issue_description):
    ticket_id = f"TCK-{int(datetime.now().timestamp())}"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO tickets (ticket_id, timestamp, issue, status) VALUES (?, ?, ?, ?)",
        (ticket_id, timestamp, issue_description, "Escalated to IT Team")
    )
    conn.commit()
    conn.close()
    return ticket_id

def get_all_tickets():
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query("SELECT * FROM tickets", conn)
    conn.close()
    return df

def ai_helpdesk_agent(user_query):
    retrieved = retrieve_relevant_issue(user_query)
    if retrieved["confidence"] < 0.3:
        return {"response": None, "resolved": False, "retrieved": retrieved}
    prompt = f"""You are a professional IT helpdesk agent.
User issue: \"{user_query}\"
KB Issue: {retrieved['issue']}
Category: {retrieved['category']}
Solution steps: {retrieved['solution']}

Respond friendly and professional. Diagnose briefly, then give numbered solution steps."""
    answer = call_ollama(prompt)
    return {"response": answer, "resolved": True, "retrieved": retrieved}

st.set_page_config(page_title="AI IT Helpdesk Agent", page_icon="🛠️")
st.title("🛠️ AI IT Helpdesk Agent")
st.caption("Agent + RAG + Tools + SQL | Powered by Ollama (local LLM) | Describe your IT issue below")

if "history" not in st.session_state:
    st.session_state.history = []

user_query = st.chat_input("Describe your IT issue...")

if user_query:
    result = ai_helpdesk_agent(user_query)
    st.session_state.history.append(("user", user_query))
    if result["resolved"]:
        st.session_state.history.append(("agent", result["response"]))
    else:
        ticket_id = create_ticket(user_query)
        msg = f"I couldn't find a confident match. I've escalated this — Ticket ID: **{ticket_id}**"
        st.session_state.history.append(("agent", msg))

for role, msg in st.session_state.history:
    with st.chat_message(role):
        st.write(msg)

st.sidebar.header("📋 Raised Tickets (from SQL database)")
tickets_df = get_all_tickets()
if not tickets_df.empty:
    st.sidebar.dataframe(tickets_df)
else:
    st.sidebar.write("No tickets yet.")

Writing app.py


## Step 10: Install Streamlit

In [11]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.5 MB/s eta 0:00:00


## Step 11: Clean Up Any Old Running Processes
Run this every time before starting a fresh deployment (especially after reconnecting on a new day). Note: this does NOT stop Ollama — Ollama needs to keep running for the app to work.

In [ ]:
!pkill -f streamlit
!pkill -f cloudflared
!pkill -f lt
!pkill -f ngrok

## Step 12: Download Cloudflare Tunnel (no signup needed)

In [14]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

## Step 13: Start the Streamlit App (in background)

In [15]:
!streamlit run app.py --server.headless=true &>/content/logs.txt &
import time
time.sleep(5)
print("Streamlit started. Now run the next cell to get your public link.")

Streamlit started. Now run the next cell to get your public link.


## Step 14: Get Your Public App Link
Run this cell and wait for a line containing a link like `https://xxxx-xxxx.trycloudflare.com` — that's your live app link. Open it in a new browser tab. Keep this cell running while you use/demo the app.

In [16]:
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-09-12T07:59:48Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-12T07:59:48Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-12T07:59:52Z INF +--------------------------------------------------------------------------------------------+
2026-09-12T07:59:52Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-12T07:59:52Z INF |  https://acm-ordered-apply-cove.trycloudflare.com     